<a href="https://colab.research.google.com/github/Swaroop2108/Deep-Learning/blob/main/DL_Skill2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End VisionPort Architecture
This notebook demonstrates a complete end-to-end pipeline using **FastAPI** for the client-facing API, **Triton Inference Server** for serving models, and **TensorRT** for optimizing deep learning models for high-performance GPU execution.

In [ ]:
# Step 1: Check System and GPU Environment
import sys
import torch
import platform

print(f"Python Version: {sys.version}")
print(f"Operating System: {platform.platform()}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Python Version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Operating System: Linux-6.6.159+-x86_64-with-glibc2.39
CUDA Available: False


## Step 2: Install Pipeline Dependencies

We will install FastAPI, Uvicorn (to serve FastAPI), and `tritonclient` (to communicate with our simulated or remote Triton Inference Server).

In [ ]:
!pip install -q fastapi uvicorn tritonclient[all]

## Step 3: Architecture Definition & Triton Model Config

In a production environment, our pipeline flows as follows:
1. **FastAPI Client**: Receives an image from the user, performs minimal pre-processing, and converts it to a payload.
2. **Triton Inference Server**: Orchestrates execution on the GPU. It reads incoming requests and routes them to the optimized model engine.
3. **TensorRT Engine**: A compiled, highly optimized representation of our deep learning model deployed inside Triton.

Let's generate the Triton configuration file (`config.pbtxt`) for a typical TensorRT model architecture.

In [ ]:
import os

# Define model directory structure for Triton
model_name = "visionport_model"
model_version = "1"
model_dir = f"models/{model_name}/{model_version}"
os.makedirs(model_dir, exist_ok=True)

# Generate a configuration file for Triton (config.pbtxt)
config_content = f"""name: "{model_name}"
platform: "tensorrt_plan"
max_batch_size: 8
input [
  {{
    name: "input_images"
    data_type: TYPE_FP32
    dims: [ 3, 224, 224 ]
  }}
]
output [
  {{
    name: "predictions"
    data_type: TYPE_FP32
    dims: [ 1000 ]
  }}
]
"""

with open(f"models/{model_name}/config.pbtxt", "w") as f:
    f.write(config_content)

print("Created Triton model directory structure and config.pbtxt:")
!find models/

Created Triton model directory structure and config.pbtxt:
models/
models/visionport_model
models/visionport_model/config.pbtxt
models/visionport_model/1


## Step 4: FastAPI Gateway and Triton Integration

Below, we write a mock but fully programmatic representation of the FastAPI service (`main.py`) running in our architecture. It exposes an `/infer` endpoint that accepts image requests, translates them into the binary buffers that Triton expects, sends them via Triton's Client API, and responds to the user.

In [ ]:
import numpy as np
import tritonclient.http as httpclient
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List

app = FastAPI(title="VisionPort End-to-End API")

class InferenceRequest(BaseModel):
    data: List[float] # Simulated preprocessed flat image array of 3 * 224 * 224 values

@app.post("/infer")
def run_inference(request: InferenceRequest):
    if len(request.data) != 3 * 224 * 224:
        raise HTTPException(status_code=400, detail="Invalid flat array dimensions for a (3, 224, 224) image.")

    # Prepare input payload for Triton
    input_array = np.array(request.data, dtype=np.float32).reshape(1, 3, 224, 224)

    try:
        # Set up the Triton Client connection
        client = httpclient.InferenceServerClient(url="localhost:8000")

        inputs = [httpclient.InferInput("input_images", [1, 3, 224, 224], "FP32")]
        inputs[0].set_data_from_numpy(input_array)

        outputs = [httpclient.InferRequestedOutput("predictions")]

        # In a real setup, Triton Server would be running live on port 8000
        # response = client.infer("visionport_model", inputs, outputs=outputs)
        # results = response.as_numpy("predictions")

        # Simulated return output representing TensorRT output probabilities
        simulated_results = (input_array.mean() * np.ones((1, 1000))).tolist()
        return {"status": "success", "predictions_sample": simulated_results[0][:5], "total_classes": len(simulated_results[0])}

    except Exception as e:
        return {"status": "error", "detail": str(e)}

# Instantiate and test FastAPI directly inside the notebook context
import uvicorn
import threading

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8081, log_level="info")

print("FastAPI Gateway and Triton HTTP integration successfully defined on port 8081!")

FastAPI Gateway and Triton HTTP integration successfully defined on port 8081!


## Step 5: Test the End-to-End Pipeline

We will launch our FastAPI application on a background thread and send a simulated client request containing an array representing an input image.

In [ ]:
import requests
import time

# 1. Start the FastAPI server in a background thread
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Allow a brief moment for the server to spin up
time.sleep(2)

# 2. Simulate a client request (image of shape [3, 224, 224] flattened to 150528 elements)
simulated_image = [0.5] * (3 * 224 * 224)
payload = {"data": simulated_image}

# 3. Call our FastAPI Gateway endpoint on port 8081
try:
    response = requests.post("http://127.0.0.1:8081/infer", json=payload)
    print("\n--- FastAPI Service Response ---")
    print("Status Code:", response.status_code)
    print("JSON Output:", response.json())
except Exception as e:
    print("Failed to request FastAPI server:", e)

INFO:     Started server process [544]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8081 (Press CTRL+C to quit)


INFO:     127.0.0.1:34638 - "POST /infer HTTP/1.1" 200 OK

--- FastAPI Service Response ---
Status Code: 200
JSON Output: {'status': 'success', 'predictions_sample': [0.5, 0.5, 0.5, 0.5, 0.5], 'total_classes': 1000}
